# 04 Dodatkowe analizy – walidacja modelu NLI (H4a)

## 1. Konfiguracja

In [1]:
import sys
sys.path.append("..")

import pandas as pd
import torch
from sklearn.metrics import cohen_kappa_score, f1_score
from transformers import AutoModelForSequenceClassification, AutoTokenizer

from src.config import LABELS_DIR, NLI_MODEL, PROCESSED_DIR, RESULTS_DIR
from src.nli import nli_probabilities

VALIDATION_FILES = ["r_democrats_comments_2016", "r_Conservative_comments_2016"]
BATCH_SIZE = 8
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
device = "cuda" if torch.cuda.is_available() else "cpu"
print("urządzenie:", device)

c:\Users\karol\Licencjat_programowanie\Licencjat\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


urządzenie: cuda


## 2. Etykiety i teksty (`labels/stance_to_label.csv`: `id`, `term`, `text`, `label`)

In [ ]:
texts = pd.concat(pd.read_parquet(PROCESSED_DIR / "clean" / f"{file}.parquet", columns=["id", "text"])
                  for file in VALIDATION_FILES)
# plik zapisany z Excela: separator ";" i kodowanie cp1252; tekst bierzemy z parquetu, nie z Excela
labels = pd.read_csv(LABELS_DIR / "stance_to_label.csv", sep=";", encoding="cp1252", dtype={"id": str})
data = labels[["id", "term", "label"]].merge(texts, on="id")
print("etykiet:", len(labels), "| znalezionych tekstów:", len(data))

## 3. Model

In [3]:
tokenizer = AutoTokenizer.from_pretrained(NLI_MODEL)
model = AutoModelForSequenceClassification.from_pretrained(NLI_MODEL).to(device).eval()

W1110 09:26:26.133000 27172 Lib\site-packages\torch\distributed\elastic\multiprocessing\redirects.py:29] NOTE: Redirects are currently not supported in Windows or MacOs.
Loading weights: 100%|██████████| 394/394 [00:00<00:00, 659.54it/s]


## 4. Kappa Cohena i F1 dla obu wariantów

In [ ]:
probabilities = nli_probabilities(data["text"].tolist(), data["term"].tolist(), model, tokenizer, device, BATCH_SIZE)
scores = []
for variant, column in (("główny: 2 klasy", "critical"), ("porównawczy: 3 klasy", "critical_3")):
    scores.append({"wariant": variant, "n": len(data),
                   "kappa": cohen_kappa_score(data["label"], probabilities[column]),
                   "f1": f1_score(data["label"], probabilities[column])})
scores = pd.DataFrame(scores)
scores.to_csv(RESULTS_DIR / "validation_stance.csv", index=False)
scores.round(3)

## 5. Diagnoza: odsetek krytyki i macierz pomyłek (wariant główny), osobno dla `own` i `out`

In [ ]:
data["predicted"] = probabilities["critical"].astype(int).values
print("odsetek krytyki – etykiety:", round(data["label"].mean(), 3), "| model:", round(data["predicted"].mean(), 3))
print(pd.crosstab(data["label"], data["predicted"], rownames=["etykieta"], colnames=["model"]))

strata = pd.read_csv(PROCESSED_DIR / "validation" / "stance_to_label_strata.csv", dtype={"id": str})
data = data.merge(strata[["id", "target"]], on="id")
for target, group in data.groupby("target"):
    print()
    print(f"{target} (n = {len(group)}) – etykiety:", round(group["label"].mean(), 3), "| model:", round(group["predicted"].mean(), 3))
    print(pd.crosstab(group["label"], group["predicted"], rownames=["etykieta"], colnames=["model"]))